# doc-structure-mcp ?????? OCR + bbox ??

?????????????? `test_page1.png` ? `test_page2.pdf`??? Colab T4 GPU ?????? PaddleOCR 3.6 ??????????? OCR JSON?bbox ?????????????????????

???????????? `!nvidia-smi` ?? GPU???? Colab ?? torch??? CUDA/NCCL ???

| ?? | ???? | ???? | ?? |
|---|---|---|---|
| `mobile` | `PP-OCRv5_mobile_det` | `PP-OCRv5_mobile_rec` | ???? |
| `mid` | `PP-OCRv5_server_det` | `PP-OCRv5_mobile_rec` | ???????? bbox ?? |
| `server` | `PP-OCRv5_server_det` | `PP-OCRv5_server_rec` | ?? server ?? |


In [ ]:
# 1. ?? GPU????? Tesla T4 / GPU ???
!nvidia-smi


In [ ]:
# 2. ?? torch??? PaddleOCR/PaddleX import ??? Colab ?? torch ? CUDA/NCCL ???
!pip uninstall -y torch torchvision torchaudio torchtext torchdata


In [ ]:
# 3. ?? PaddleOCR ??
import os, sys, platform, subprocess
os.environ['PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK'] = 'True'

# ??????????????? GPU??????? GPU Paddle??? libcuda.so.1 ???
if subprocess.run('nvidia-smi', shell=True).returncode != 0:
    raise RuntimeError('?????? GPU/libcuda???????? -> ?????????? T4 GPU??????/???????????????')

!pip uninstall -y paddlepaddle paddlepaddle-gpu paddleocr paddlex >/dev/null 2>&1
!python -m pip install -q paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install -q "paddleocr[doc-parser]==3.6.0" pymupdf opencv-python-headless pillow

# PPStructureV3 ???? PaddleX ? ocr extra ??????????????? paddlex[ocr]?
import importlib.metadata as metadata
paddlex_version = metadata.version('paddlex')
print('Install PaddleX OCR extra:', paddlex_version)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', f'paddlex[ocr]=={paddlex_version}'])

import paddle
print('Python:', sys.version)
print('Platform:', platform.platform())
print('Paddle:', paddle.__version__)
print('CUDA compiled:', paddle.is_compiled_with_cuda())
if not paddle.is_compiled_with_cuda():
    raise RuntimeError('Paddle ???? CUDA ????????')
!nvidia-smi


In [ ]:
# 4. ????????
# ??????? test_page1.png??????? test_page2.pdf?
from google.colab import files
from pathlib import Path

uploaded = files.upload()
INPUT_FILE = list(uploaded.keys())[0]
CASE_NAME = Path(INPUT_FILE).stem.replace(' ', '_').replace('(', '').replace(')', '') + '_three_models_v1'
DPI = 220
print('INPUT_FILE:', INPUT_FILE)
print('CASE_NAME:', CASE_NAME)
print('DPI:', DPI)


In [ ]:
# 5. ??????
MODEL_SPECS = [
    {'name': 'mobile', 'det': 'PP-OCRv5_mobile_det', 'rec': 'PP-OCRv5_mobile_rec', 'purpose': '????'},
    {'name': 'mid', 'det': 'PP-OCRv5_server_det', 'rec': 'PP-OCRv5_mobile_rec', 'purpose': '???????? bbox ??'},
    {'name': 'server', 'det': 'PP-OCRv5_server_det', 'rec': 'PP-OCRv5_server_rec', 'purpose': '?? server ??'},
]
for spec in MODEL_SPECS:
    print(spec['name'], spec['det'], '+', spec['rec'], '-', spec['purpose'])


In [ ]:
# 6. ??????
import gc, json, time, zipfile
from pathlib import Path

import cv2
import fitz
from PIL import Image, ImageDraw, ImageFont
from paddleocr import PaddleOCR


def render_input(input_path, dpi):
    input_path = Path(input_path)
    if input_path.suffix.lower() == '.pdf':
        doc = fitz.open(str(input_path))
        pages = []
        matrix = fitz.Matrix(dpi / 72, dpi / 72)
        for index, page in enumerate(doc, start=1):
            pix = page.get_pixmap(matrix=matrix, alpha=False)
            page_png = Path(f'{input_path.stem}_page_{index:03d}.png')
            pix.save(str(page_png))
            image = cv2.imread(str(page_png))
            pages.append({'page_no': index, 'image_path': page_png, 'image': image})
        doc.close()
        return pages
    image = cv2.imread(str(input_path))
    if image is None:
        raise RuntimeError(f'Cannot read image: {input_path}')
    return [{'page_no': 1, 'image_path': input_path, 'image': image}]


def extract_lines(predict_result):
    lines = []
    for result in predict_result:
        if result is None:
            continue
        if isinstance(result, dict):
            texts = result.get('rec_texts', [])
            scores = result.get('rec_scores', [])
            polys = result.get('dt_polys', [])
        else:
            texts = getattr(result, 'rec_texts', [])
            scores = getattr(result, 'rec_scores', [])
            polys = getattr(result, 'dt_polys', [])
        for index, text in enumerate(texts):
            text = str(text).strip()
            if not text:
                continue
            score = float(scores[index]) if index < len(scores) else 0.0
            poly = polys[index] if index < len(polys) else None
            if poly is None:
                continue
            points = []
            for point in poly:
                if len(point) >= 2:
                    points.append([float(point[0]), float(point[1])])
            if len(points) < 4:
                continue
            xs = [p[0] for p in points]
            ys = [p[1] for p in points]
            lines.append({'text': text, 'confidence': score, 'bbox': points, 'x1': min(xs), 'y1': min(ys), 'x2': max(xs), 'y2': max(ys)})
    return lines


def rect(points):
    xs = [float(p[0]) for p in points]
    ys = [float(p[1]) for p in points]
    return min(xs), min(ys), max(xs), max(ys)


def cluster_rows(lines, y_threshold=18):
    items = []
    for line in lines:
        x1, y1, x2, y2 = rect(line['bbox'])
        items.append({**line, 'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2, 'cy': (y1 + y2) / 2})
    items.sort(key=lambda item: (item['y1'], item['x1']))
    rows = []
    for item in items:
        if not rows:
            rows.append([item])
            continue
        last_cy = sum(x['cy'] for x in rows[-1]) / len(rows[-1])
        if abs(item['cy'] - last_cy) <= y_threshold:
            rows[-1].append(item)
        else:
            rows.append([item])
    for row in rows:
        row.sort(key=lambda item: item['x1'])
    return rows


def draw_boxes(image_path, page, output_path):
    image = Image.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(image)
    try:
        font = ImageFont.truetype('arial.ttf', 16)
    except Exception:
        font = None
    for index, line in enumerate(page['lines'], start=1):
        x1, y1, x2, y2 = rect(line['bbox'])
        draw.rectangle((x1, y1, x2, y2), outline='red', width=2)
        draw.text((x1, max(0, y1 - 18)), str(index), fill='blue', font=font)
    image.save(output_path)


def evaluate_page(page):
    lines = page['lines']
    confs = [float(line.get('confidence', 0)) for line in lines]
    low = [c for c in confs if c < 0.75]
    tall = []
    for line in lines:
        x1, y1, x2, y2 = rect(line['bbox'])
        if (y2 - y1) > max(1, x2 - x1) * 2 and (y2 - y1) > 40:
            tall.append(line)
    rows = cluster_rows(lines)
    warnings = []
    if len(lines) < 8:
        warnings.append('line_count_low')
    if lines and len(low) / len(lines) > 0.25:
        warnings.append('many_low_confidence_lines')
    if lines and len(tall) / len(lines) > 0.2:
        warnings.append('many_tall_boxes_check_vertical_or_merged_text')
    if rows and max(len(row) for row in rows) >= 6:
        warnings.append('dense_row_maybe_table_or_bad_line_merge')
    avg = sum(confs) / len(confs) if confs else 0
    if not lines:
        grade = 'D'
    elif avg >= 0.90 and len(low) <= 2 and len(warnings) <= 1:
        grade = 'A'
    elif avg >= 0.82 and len(warnings) <= 3:
        grade = 'B'
    elif avg >= 0.70:
        grade = 'C'
    else:
        grade = 'D'
    sorted_text = '\n'.join(' '.join(item['text'] for item in row) for row in rows)
    return {'page_no': page['page_no'], 'line_count': len(lines), 'row_count': len(rows), 'avg_confidence': round(avg, 4), 'low_confidence_count': len(low), 'tall_box_count': len(tall), 'warnings': warnings, 'grade': grade, 'sorted_text': sorted_text}


def run_model(spec, rendered_pages, case_dir):
    model_dir = case_dir / spec['name']
    model_dir.mkdir(parents=True, exist_ok=True)
    print('\n' + '=' * 80)
    print(f"RUN {spec['name']}: {spec['det']} + {spec['rec']}")
    print('=' * 80)

    load_start = time.time()
    ocr = PaddleOCR(text_detection_model_name=spec['det'], text_recognition_model_name=spec['rec'], use_textline_orientation=True, lang='ch', text_det_thresh=0.3, text_det_box_thresh=0.5)
    load_ms = int((time.time() - load_start) * 1000)

    pages = []
    total_start = time.time()
    for page in rendered_pages:
        page_start = time.time()
        predict_result = list(ocr.predict(page['image']))
        lines = extract_lines(predict_result)
        lines.sort(key=lambda item: (item['y1'], item['x1']))
        image_name = f"page_{page['page_no']:03d}.png"
        image_out = model_dir / image_name
        cv2.imwrite(str(image_out), page['image'])
        pages.append({'page_no': page['page_no'], 'image_file': image_name, 'width': int(page['image'].shape[1]), 'height': int(page['image'].shape[0]), 'duration_ms': int((time.time() - page_start) * 1000), 'lines': lines, 'plain_text': '\n'.join(line['text'] for line in lines)})
        print(f"page {page['page_no']}: {len(lines)} lines, {pages[-1]['duration_ms']} ms")

    payload = {'case': CASE_NAME, 'model_name': spec['name'], 'purpose': spec['purpose'], 'input_file': INPUT_FILE, 'dpi': DPI, 'model': {'det': spec['det'], 'rec': spec['rec']}, 'load_duration_ms': load_ms, 'total_duration_ms': int((time.time() - total_start) * 1000), 'pages': pages}
    (model_dir / 'ocr_result.json').write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')

    summaries = []
    for page in pages:
        image_path = model_dir / page['image_file']
        box_image = model_dir / f"page_{page['page_no']:03d}_boxes.png"
        draw_boxes(image_path, page, box_image)
        summary = evaluate_page(page)
        summary['box_image'] = box_image.name
        summaries.append(summary)
        (model_dir / f"page_{page['page_no']:03d}_sorted.txt").write_text(summary['sorted_text'], encoding='utf-8')

    report = {'case': CASE_NAME, 'model_name': spec['name'], 'purpose': spec['purpose'], 'input_file': INPUT_FILE, 'model': {'det': spec['det'], 'rec': spec['rec']}, 'dpi': DPI, 'load_duration_ms': load_ms, 'total_duration_ms': payload['total_duration_ms'], 'pages': summaries}
    (model_dir / 'evaluation.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')

    md = [f"# OCR Evaluation: {CASE_NAME} / {spec['name']}", '', f"- purpose: `{spec['purpose']}`", f"- input: `{INPUT_FILE}`", f"- model: `{spec['det']}` + `{spec['rec']}`", f"- dpi: `{DPI}`", f"- load_duration_ms: `{load_ms}`", f"- total_duration_ms: `{payload['total_duration_ms']}`", '', '| Page | Grade | Lines | Rows | Avg Conf | Low Conf | Tall Boxes | Warnings | Box Image |', '|---:|---|---:|---:|---:|---:|---:|---|---|']
    for page in summaries:
        md.append(f"| {page['page_no']} | {page['grade']} | {page['line_count']} | {page['row_count']} | {page['avg_confidence']} | {page['low_confidence_count']} | {page['tall_box_count']} | {', '.join(page['warnings']) or '-'} | {page['box_image']} |")
    md.extend(['', '## Sorted Text Preview', ''])
    for page in summaries:
        md.extend([f"### Page {page['page_no']}", '```text', page['sorted_text'][:3000], '```', ''])
    (model_dir / 'evaluation.md').write_text('\n'.join(md), encoding='utf-8')

    del ocr
    gc.collect()
    return report


In [ ]:
# 7. ??????????????
CASE_DIR = Path('/content/ocr_lab_outputs') / CASE_NAME
CASE_DIR.mkdir(parents=True, exist_ok=True)
rendered_pages = render_input(INPUT_FILE, DPI)
print('rendered pages:', [(p['page_no'], p['image'].shape[1], p['image'].shape[0]) for p in rendered_pages])

all_reports = []
for spec in MODEL_SPECS:
    try:
        all_reports.append(run_model(spec, rendered_pages, CASE_DIR))
    except Exception as exc:
        print(f"ERROR in {spec['name']}: {exc}")
        all_reports.append({'case': CASE_NAME, 'model_name': spec['name'], 'purpose': spec['purpose'], 'input_file': INPUT_FILE, 'model': {'det': spec['det'], 'rec': spec['rec']}, 'error': str(exc), 'pages': []})

(CASE_DIR / 'model_compare.json').write_text(json.dumps(all_reports, ensure_ascii=False, indent=2), encoding='utf-8')
print('done:', CASE_DIR)


In [ ]:
# 8. ???????
compare_md = [
    f'# Model Compare: {CASE_NAME}', '',
    f'- input: `{INPUT_FILE}`',
    f'- dpi: `{DPI}`', '',
    '| Model | Purpose | Det | Rec | Load ms | Infer ms | Grade | Lines | Avg Conf | Low Conf | Tall Boxes | Error |',
    '|---|---|---|---|---:|---:|---|---:|---:|---:|---:|---|',
]
for report in all_reports:
    if report.get('error'):
        compare_md.append(
            f"| {report['model_name']} | {report['purpose']} | {report['model']['det']} | {report['model']['rec']} | - | - | - | - | - | - | - | {report['error']} |"
        )
        continue
    page0 = report['pages'][0] if report.get('pages') else {}
    compare_md.append(
        f"| {report['model_name']} | {report['purpose']} | {report['model']['det']} | {report['model']['rec']} | "
        f"{report.get('load_duration_ms', '-')} | {report.get('total_duration_ms', '-')} | "
        f"{page0.get('grade', '-')} | {page0.get('line_count', '-')} | {page0.get('avg_confidence', '-')} | "
        f"{page0.get('low_confidence_count', '-')} | {page0.get('tall_box_count', '-')} | - |"
    )

compare_md.extend([
    '',
    '## How to read',
    '',
    '- `mobile -> mid`: mainly compare detection boxes, missed boxes, fragmented boxes, and layout order.',
    '- `mid -> server`: mainly compare confidence, wrong characters, and low-confidence blocks.',
    '- Do not trust Grade alone. Manually inspect each model folder: `page_001_boxes.png` and `page_001_sorted.txt`.',
])
compare_text = "\n".join(compare_md)
(CASE_DIR / 'model_compare.md').write_text(compare_text, encoding='utf-8')
print(compare_text)


In [ ]:
# 9. ????
zip_path = CASE_DIR.with_suffix('.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in CASE_DIR.rglob('*'):
        archive.write(path, path.relative_to(CASE_DIR.parent))
print('download:', zip_path)
files.download(str(zip_path))



## 10. TABLE_PROBE_SERVER_V1: ????????

???????????????? `PPStructureV3`????? server/?? pipeline ????????????

???????????? DOCX ???????

- ???? `table_res_list`
- ???? `cell_box_list`
- ???? `pred_html`
- ??????? OCR ?? `table_ocr_pred`
- cell bbox ? OCR bbox ?????????????

?????`/content/ocr_lab_outputs/<CASE_NAME>_table_probe_server_v1/`


In [ ]:
# 10A. TABLE_PROBE_SERVER_V1_DEPS - ?? PPStructureV3 ?????
# ? 3 ????????? paddleocr[doc-parser] ? paddlex[ocr]?
# ????????????????? 1 ????????? import PaddleOCR/PaddleX ??????????
import importlib.metadata as metadata

print('paddleocr:', metadata.version('paddleocr'))
print('paddlex:', metadata.version('paddlex'))

from paddleocr import PPStructureV3
print('PPStructureV3 import OK')


In [ ]:

# 10. TABLE_PROBE_SERVER_V1 - ?????????
import inspect
import json
import time
from pathlib import Path

import cv2
from PIL import Image, ImageDraw

TABLE_PROBE_NAME = CASE_NAME + '_table_probe_server_v1'
TABLE_PROBE_DIR = Path('/content/ocr_lab_outputs') / TABLE_PROBE_NAME
TABLE_PROBE_DIR.mkdir(parents=True, exist_ok=True)


def jsonable(obj):
    if obj is None or isinstance(obj, (str, int, float, bool)):
        return obj
    if isinstance(obj, dict):
        return {str(k): jsonable(v) for k, v in obj.items() if k != 'img'}
    if isinstance(obj, (list, tuple)):
        return [jsonable(v) for v in obj]
    if hasattr(obj, 'tolist'):
        return jsonable(obj.tolist())
    if hasattr(obj, 'json'):
        return jsonable(obj.json)
    return str(obj)


def make_ppstructure_v3():
    from paddleocr import PPStructureV3

    attempts = [
        {
            'use_doc_orientation_classify': False,
            'use_doc_unwarping': False,
            'use_textline_orientation': False,
            'use_formula_recognition': False,
            'use_seal_recognition': False,
            'use_table_recognition': True,
            'device': 'gpu',
        },
        {
            'use_doc_orientation_classify': False,
            'use_doc_unwarping': False,
            'use_textline_orientation': False,
            'use_formula_recognition': False,
            'use_table_recognition': True,
            'device': 'gpu',
        },
        {'device': 'gpu'},
        {},
    ]
    last_error = None
    for kwargs in attempts:
        try:
            print('Trying PPStructureV3 kwargs:', kwargs)
            return PPStructureV3(**kwargs), kwargs
        except TypeError as exc:
            last_error = exc
            print('PPStructureV3 kwargs rejected:', exc)
    raise last_error


def normalize_table_result(res):
    if hasattr(res, 'json'):
        return jsonable(res.json)
    if isinstance(res, dict):
        return jsonable(res)
    return jsonable(res)


def find_table_items(page_json):
    candidates = []
    if isinstance(page_json, dict):
        for key in ['table_res_list', 'tables', 'table_results']:
            value = page_json.get(key)
            if isinstance(value, list):
                candidates.extend(value)
    return candidates


def box_to_rect(box):
    if not box:
        return None
    if isinstance(box, dict):
        for key in ['bbox', 'box', 'cell_box']:
            if key in box:
                return box_to_rect(box[key])
    if isinstance(box, (list, tuple)):
        if len(box) == 4 and all(isinstance(v, (int, float)) for v in box):
            x1, y1, x2, y2 = [float(v) for v in box]
            return x1, y1, x2, y2
        points = []
        for item in box:
            if isinstance(item, (list, tuple)) and len(item) >= 2:
                points.append([float(item[0]), float(item[1])])
        if points:
            xs = [p[0] for p in points]
            ys = [p[1] for p in points]
            return min(xs), min(ys), max(xs), max(ys)
    return None


def collect_cell_boxes(table_item):
    boxes = []
    if not isinstance(table_item, dict):
        return boxes
    for key in ['cell_box_list', 'cell_bbox_list', 'cells']:
        value = table_item.get(key)
        if isinstance(value, list):
            for item in value:
                rect = box_to_rect(item)
                if rect:
                    boxes.append(rect)
    return boxes


def draw_table_probe(page_image, page_json, out_path):
    image = Image.fromarray(cv2.cvtColor(page_image, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(image)
    tables = find_table_items(page_json)
    for table_index, table in enumerate(tables, start=1):
        table_rect = None
        if isinstance(table, dict):
            table_rect = box_to_rect(table.get('bbox') or table.get('box') or table.get('table_bbox'))
        if table_rect:
            draw.rectangle(table_rect, outline='blue', width=4)
            draw.text((table_rect[0] + 4, table_rect[1] + 4), f'table {table_index}', fill='blue')
        for cell_rect in collect_cell_boxes(table):
            draw.rectangle(cell_rect, outline='red', width=2)
    image.save(out_path)


def summarize_structure_result(raw_pages):
    lines = [f'# Table Probe: {TABLE_PROBE_NAME}', '', f'- input: `{INPUT_FILE}`', f'- dpi: `{DPI}`', '']
    lines.extend(['| Page | Tables | Cell boxes | Has HTML | Has table OCR | Keys |', '|---:|---:|---:|---|---|---|'])
    for page_index, page_json in enumerate(raw_pages, start=1):
        tables = find_table_items(page_json)
        cell_count = sum(len(collect_cell_boxes(t)) for t in tables)
        has_html = any(isinstance(t, dict) and bool(t.get('pred_html') or t.get('html')) for t in tables)
        has_table_ocr = any(isinstance(t, dict) and bool(t.get('table_ocr_pred')) for t in tables)
        keys = ', '.join(sorted(page_json.keys())) if isinstance(page_json, dict) else type(page_json).__name__
        lines.append(f'| {page_index} | {len(tables)} | {cell_count} | {has_html} | {has_table_ocr} | `{keys}` |')
    lines.extend([
        '',
        '## ????',
        '',
        '- ?? `Cell boxes > 0`??????? OCR line bbox ? cell bbox ??????/?????????',
        '- ???? `pred_html` ?? cell bbox??????????????????????????',
        '- ???? table result????? PPStructureV3 ????????? TableRecognitionPipelineV2?',
    ])
    return '\n'.join(lines)


In [ ]:

# 11. TABLE_PROBE_SERVER_V1 - ?? PPStructureV3 ????
# ???????? pipeline??????? mobile/mid/server ??? OCR pipeline?
# ??? table/cell ????????????????????????

table_probe_report = {
    'case': TABLE_PROBE_NAME,
    'input_file': INPUT_FILE,
    'dpi': DPI,
    'pipeline': 'PPStructureV3',
    'pages': [],
}

try:
    structure_pipeline, structure_kwargs = make_ppstructure_v3()
    table_probe_report['pipeline_kwargs'] = structure_kwargs
    start = time.perf_counter()
    raw_pages = []

    for page in rendered_pages:
        page_no = page['page_no']
        page_dir = TABLE_PROBE_DIR / f'page_{page_no:03d}'
        page_dir.mkdir(parents=True, exist_ok=True)
        page_img_path = page_dir / 'page.png'
        cv2.imwrite(str(page_img_path), page['image'])

        print('PPStructureV3 predict:', page_img_path)
        output = structure_pipeline.predict(str(page_img_path))
        page_results = []
        for res_index, res in enumerate(output):
            page_json = normalize_table_result(res)
            page_results.append(page_json)
            (page_dir / f'structure_res_{res_index:03d}.json').write_text(
                json.dumps(page_json, ensure_ascii=False, indent=2), encoding='utf-8'
            )
            if hasattr(res, 'save_to_json'):
                res.save_to_json(save_path=str(page_dir / 'official_json'))
            if hasattr(res, 'save_to_markdown'):
                res.save_to_markdown(save_path=str(page_dir / 'official_markdown'))
            if hasattr(res, 'save_to_img'):
                res.save_to_img(save_path=str(page_dir / 'official_img'))

        page_json = page_results[0] if page_results else {}
        draw_table_probe(page['image'], page_json, page_dir / 'table_boxes.png')
        raw_pages.append(page_json)
        table_probe_report['pages'].append({
            'page_no': page_no,
            'image_file': str(page_img_path.relative_to(TABLE_PROBE_DIR)),
            'raw_result_files': [str((page_dir / f'structure_res_{i:03d}.json').relative_to(TABLE_PROBE_DIR)) for i in range(len(page_results))],
            'table_boxes_image': str((page_dir / 'table_boxes.png').relative_to(TABLE_PROBE_DIR)),
            'table_count': len(find_table_items(page_json)),
            'cell_box_count': sum(len(collect_cell_boxes(t)) for t in find_table_items(page_json)),
        })

    table_probe_report['duration_ms'] = int((time.perf_counter() - start) * 1000)
    (TABLE_PROBE_DIR / 'table_probe_report.json').write_text(
        json.dumps(table_probe_report, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    summary = summarize_structure_result(raw_pages)
    (TABLE_PROBE_DIR / 'table_probe_summary.md').write_text(summary, encoding='utf-8')
    print(summary)

except Exception as exc:
    table_probe_report['error'] = repr(exc)
    (TABLE_PROBE_DIR / 'table_probe_report.json').write_text(
        json.dumps(table_probe_report, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    (TABLE_PROBE_DIR / 'table_probe_error.txt').write_text(repr(exc), encoding='utf-8')
    print('TABLE_PROBE_SERVER_V1 ERROR:', repr(exc))
    raise


In [ ]:

# 12. TABLE_PROBE_SERVER_V1 - ????????????
# ???????? OCR zip ??????????? table probe ????

table_zip_path = TABLE_PROBE_DIR.with_suffix('.zip')
with zipfile.ZipFile(table_zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in TABLE_PROBE_DIR.rglob('*'):
        archive.write(path, path.relative_to(TABLE_PROBE_DIR.parent))
print('download table probe:', table_zip_path)
files.download(str(table_zip_path))
